In [29]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn as nn

In [52]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [31]:
# Function to transform the images to tensors and normalize them
transform_to_tensor = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]
)

In [32]:
# Load the CIFAR-10 dataset
train_set = torchvision.datasets.CIFAR10(root='../data', train = True, download = True, transform=transform_to_tensor)
test_set = torchvision.datasets.CIFAR10(root='../data', train = False, download = True, transform=transform_to_tensor)

In [33]:
# Define the hyperparameters
num_classes = 10
batch_size = 64
img_size = 32
num_channel = 3
patch_size = 8
num_patches = (img_size // patch_size) ** 2
embedding_dim = 64
attn_heads = 4
transformer_blocks = 4
mlp_hidden_nodes = 2 * embedding_dim
learning_rate = 0.001
epochs = 5

In [34]:
# Create data loaders for the training and test sets
train_loader = DataLoader(dataset = train_set, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(dataset = test_set, batch_size = batch_size, shuffle = True)

In [35]:
# Define the PatchEmbedding class
class PatchEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_channels = num_channel, out_channels = embedding_dim, kernel_size = patch_size, stride = patch_size)

    def forward(self, X):
        out = self.patch_embed(X)
        out = out.flatten(2)
        out = out.transpose(-2, -1)

        return out

### Experiment with the handcoded MultiHeadAttention Module

In [36]:
from src.attention import MultiHeadAttention

In [37]:
# Define the TransformerEncoder class by using the handcoded MultiHeadAttention module
class TransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        self.multihead_attention = MultiHeadAttention(embedding_dim, embedding_dim, num_heads=attn_heads)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, mlp_hidden_nodes),
            nn.GELU(),
            nn.Linear(mlp_hidden_nodes, embedding_dim)            
        )

    def forward(self, X):
        residual1 = X
        out = self.layer_norm1(X)
        out = self.multihead_attention(out)
        out = out + residual1

        residual2 = out
        out = self.layer_norm2(out)
        out = self.mlp(out)
        out = out + residual2

        return out

In [38]:
# Define the MLPHead class
class MLPHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm = nn.LayerNorm(embedding_dim)
        self.mlp_layer = nn.Linear(embedding_dim, num_classes)

    def forward(self, X):
        out = self.layer_norm(X)
        out = self.mlp_layer(out)

        return out

In [39]:
# Define the VisionTransformer class
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embedding = PatchEmbedding()
        self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
        self.position_embedding = nn.Parameter(torch.randn(1, 1 + num_patches, embedding_dim))
        self.transformer_block = nn.Sequential(*[TransformerEncoder() for _ in range(transformer_blocks)])
        self.mlp_head = MLPHead()

    def forward(self, X):
        out = self.patch_embedding(X)
        B = out.size()[0]
        class_tokens = self.cls_token.expand(B, -1, -1)
        out = torch.cat((class_tokens, out), dim = 1)
        out = out + self.position_embedding
        out = self.transformer_block(out)
        out = out[:, 0]
        out = self.mlp_head(out)

        return out


In [40]:
# Define the device, model, optimizer, and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisionTransformer().to(device)
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [41]:
# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct_epochs = 0
    total_epochs = 0
    print(f"Epoch {epoch + 1}/{epochs}")

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        corrects = (preds == labels).sum().item()
        accuracy = 100 * corrects / labels.size(0)
        correct_epochs += corrects
        total_epochs += labels.size(0)

        if (i + 1) % 100 == 0:
            print(f"Batch {i + 1}/{len(train_loader)}, Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%")

    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = 100 * correct_epochs / total_epochs
    print(f"Epoch {epoch + 1} Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")
    

Epoch 1/5
Batch 100/782, Loss: 2.1895, Accuracy: 12.50%
Batch 200/782, Loss: 1.6837, Accuracy: 42.19%
Batch 300/782, Loss: 1.8306, Accuracy: 34.38%
Batch 400/782, Loss: 1.6245, Accuracy: 46.88%
Batch 500/782, Loss: 1.5900, Accuracy: 46.88%
Batch 600/782, Loss: 1.4139, Accuracy: 45.31%
Batch 700/782, Loss: 1.7013, Accuracy: 40.62%
Epoch 1 Loss: 1.6717, Accuracy: 39.43%
Epoch 2/5
Batch 100/782, Loss: 1.2243, Accuracy: 56.25%
Batch 200/782, Loss: 1.1890, Accuracy: 57.81%
Batch 300/782, Loss: 1.3759, Accuracy: 50.00%
Batch 400/782, Loss: 1.4447, Accuracy: 53.12%
Batch 500/782, Loss: 1.2702, Accuracy: 56.25%
Batch 600/782, Loss: 1.4068, Accuracy: 50.00%
Batch 700/782, Loss: 1.2774, Accuracy: 57.81%
Epoch 2 Loss: 1.3887, Accuracy: 50.25%
Epoch 3/5
Batch 100/782, Loss: 1.3323, Accuracy: 53.12%
Batch 200/782, Loss: 1.2445, Accuracy: 54.69%
Batch 300/782, Loss: 1.4902, Accuracy: 37.50%
Batch 400/782, Loss: 1.4997, Accuracy: 50.00%
Batch 500/782, Loss: 1.3897, Accuracy: 51.56%
Batch 600/782, Los

In [42]:
# Evaluate the model on the test set
model.eval()
total_correct = 0
total_samples = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        corrects = (preds == labels).sum().item()
        total_correct += corrects
        total_samples += labels.size(0)

test_accuracy = 100 * total_correct / total_samples
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 55.95%


### Experiment with the Built-in nn.MultiheadAttention Module

In [43]:
# Define the TransformerEncoder class by using the built-in nn.MultiheadAttention module
class TransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        self.multihead_attention = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=attn_heads, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, mlp_hidden_nodes),
            nn.GELU(),
            nn.Linear(mlp_hidden_nodes, embedding_dim)            
        )

    def forward(self, X):
        residual1 = X
        out = self.layer_norm1(X)
        out = self.multihead_attention(out, out, out)[0]
        out = out + residual1

        residual2 = out
        out = self.layer_norm2(out)
        out = self.mlp(out)
        out = out + residual2

        return out

In [44]:
# Define the VisionTransformer class
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embedding = PatchEmbedding()
        self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
        self.position_embedding = nn.Parameter(torch.randn(1, 1 + num_patches, embedding_dim))
        self.transformer_block = nn.Sequential(*[TransformerEncoder() for _ in range(transformer_blocks)])
        self.mlp_head = MLPHead()

    def forward(self, X):
        out = self.patch_embedding(X)
        B = out.size()[0]
        class_tokens = self.cls_token.expand(B, -1, -1)
        out = torch.cat((class_tokens, out), dim = 1)
        out = out + self.position_embedding
        out = self.transformer_block(out)
        out = out[:, 0]
        out = self.mlp_head(out)

        return out


In [ ]:
# Define the model, optimizer, and loss function
model = VisionTransformer().to(device)
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [46]:
# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct_epochs = 0
    total_epochs = 0
    print(f"Epoch {epoch + 1}/{epochs}")

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        corrects = (preds == labels).sum().item()
        accuracy = 100 * corrects / labels.size(0)
        correct_epochs += corrects
        total_epochs += labels.size(0)

        if (i + 1) % 100 == 0:
            print(f"Batch {i + 1}/{len(train_loader)}, Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%")

    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = 100 * correct_epochs / total_epochs
    print(f"Epoch {epoch + 1} Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")
    

Epoch 1/5
Batch 100/782, Loss: 1.9646, Accuracy: 32.81%
Batch 200/782, Loss: 1.8285, Accuracy: 29.69%
Batch 300/782, Loss: 1.6604, Accuracy: 35.94%
Batch 400/782, Loss: 1.5199, Accuracy: 51.56%
Batch 500/782, Loss: 1.4241, Accuracy: 39.06%
Batch 600/782, Loss: 1.4255, Accuracy: 46.88%
Batch 700/782, Loss: 1.5399, Accuracy: 45.31%
Epoch 1 Loss: 1.6506, Accuracy: 40.18%
Epoch 2/5
Batch 100/782, Loss: 1.1638, Accuracy: 56.25%
Batch 200/782, Loss: 1.2349, Accuracy: 48.44%
Batch 300/782, Loss: 1.2868, Accuracy: 53.12%
Batch 400/782, Loss: 1.2653, Accuracy: 53.12%
Batch 500/782, Loss: 1.4736, Accuracy: 43.75%
Batch 600/782, Loss: 1.2837, Accuracy: 54.69%
Batch 700/782, Loss: 1.3183, Accuracy: 53.12%
Epoch 2 Loss: 1.3724, Accuracy: 50.65%
Epoch 3/5
Batch 100/782, Loss: 1.3199, Accuracy: 50.00%
Batch 200/782, Loss: 1.2427, Accuracy: 56.25%
Batch 300/782, Loss: 1.1826, Accuracy: 65.62%
Batch 400/782, Loss: 1.4012, Accuracy: 50.00%
Batch 500/782, Loss: 1.3730, Accuracy: 46.88%
Batch 600/782, Los

In [47]:
# Evaluate the model on the test set
model.eval()
total_correct = 0
total_samples = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        corrects = (preds == labels).sum().item()
        total_correct += corrects
        total_samples += labels.size(0)

test_accuracy = 100 * total_correct / total_samples
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 56.64%


### Test the vit module

In [48]:
from src.vit import VisionTransformer

In [49]:
# Define the device, model, optimizer, and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisionTransformer(num_channel, patch_size, embedding_dim, mlp_hidden_nodes, attn_heads, transformer_blocks, num_patches, num_classes).to(device)
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [50]:
# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct_epochs = 0
    total_epochs = 0
    print(f"Epoch {epoch + 1}/{epochs}")

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        corrects = (preds == labels).sum().item()
        accuracy = 100 * corrects / labels.size(0)
        correct_epochs += corrects
        total_epochs += labels.size(0)

        if (i + 1) % 100 == 0:
            print(f"Batch {i + 1}/{len(train_loader)}, Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%")

    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = 100 * correct_epochs / total_epochs
    print(f"Epoch {epoch + 1} Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")

Epoch 1/5
Batch 100/782, Loss: 1.8213, Accuracy: 45.31%
Batch 200/782, Loss: 1.6978, Accuracy: 37.50%
Batch 300/782, Loss: 1.5716, Accuracy: 42.19%
Batch 400/782, Loss: 1.4945, Accuracy: 51.56%
Batch 500/782, Loss: 1.7032, Accuracy: 42.19%
Batch 600/782, Loss: 1.5906, Accuracy: 34.38%
Batch 700/782, Loss: 1.4024, Accuracy: 51.56%
Epoch 1 Loss: 1.6598, Accuracy: 39.98%
Epoch 2/5
Batch 100/782, Loss: 1.3564, Accuracy: 50.00%
Batch 200/782, Loss: 1.2341, Accuracy: 51.56%
Batch 300/782, Loss: 1.3163, Accuracy: 51.56%
Batch 400/782, Loss: 1.2493, Accuracy: 46.88%
Batch 500/782, Loss: 1.4546, Accuracy: 45.31%
Batch 600/782, Loss: 1.0458, Accuracy: 64.06%
Batch 700/782, Loss: 1.3331, Accuracy: 53.12%
Epoch 2 Loss: 1.3842, Accuracy: 50.08%
Epoch 3/5
Batch 100/782, Loss: 1.3453, Accuracy: 50.00%
Batch 200/782, Loss: 1.2626, Accuracy: 50.00%
Batch 300/782, Loss: 1.1401, Accuracy: 53.12%
Batch 400/782, Loss: 1.1507, Accuracy: 50.00%
Batch 500/782, Loss: 1.1736, Accuracy: 57.81%
Batch 600/782, Los

In [51]:
# Evaluate the model on the test set
model.eval()
total_correct = 0
total_samples = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        corrects = (preds == labels).sum().item()
        total_correct += corrects
        total_samples += labels.size(0)

test_accuracy = 100 * total_correct / total_samples
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 56.70%
